# Multi-Agency Ridership Data Standardization and Stop-Level Aggregation Workflow


This notebook processes ridership data from multiple transit agencies to create representative stop-level datasets for weekdays, Saturdays, and Sundays. For each agency, the workflow generally includes:

- Adding day type based on the service date (Weekday, Saturday, Sunday), with holidays normalized or removed as needed.
- Cleaning and renaming columns to ensure consistency across datasets (stop IDs, stop names, and ridership fields).
- Summing boardings and alightings across directions at each stop, and aggregating multiple time periods where applicable.
- Calculating weighted averages when ridership is reported over multiple periods, using the number of service days as weights.
- Setting a ridership basis flag to indicate that values represent reported daily boardings and/or alightings.
- Computing final average ridership per stop, and day type, producing clean, representative stop-level datasets ready for modeling, analysis, or comparison across agencies.

This approach standardizes ridership data across agencies with differing data formats, missing values, and service coverage, enabling consistent cross-agency analysis.

In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
bart_ridership = pd.read_excel(f"{GCS_FILE_PATH}/bart_ridership.xlsx")
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")
caltrain_ridership = pd.read_excel(f"{GCS_FILE_PATH}/caltrain_ridership.xlsx")
culver_citybus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/culver_citybus_ridership.xlsx")
foothill_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/foothill_transit_ridership.xlsx")
fresno_area_express_ridership = pd.read_excel(f"{GCS_FILE_PATH}/fresno_area_express_ridership.xlsx")
gold_coast_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/gold_coast_transit_ridership.xlsx")
golden_gate_park_shuttle_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_park_shuttle_ridership.xlsx")
golden_gate_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_transit_ridership.xlsx")
long_beach_transit_ridership=  pd.read_excel(f"{GCS_FILE_PATH}/long_beach_transit_ridership.xlsx")
octa_ridership = pd.read_excel(f"{GCS_FILE_PATH}/octa_ridership.xlsx")
omni_trans_ridership= pd.read_excel(f"{GCS_FILE_PATH}/omni_trans_ridership.xlsx")
riverside_transit_ridership= pd.read_excel(f"{GCS_FILE_PATH}/riverside_transit_ridership.xlsx")
sacrt_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sacrt_bus_ridership.xlsx")
samtrans_ridership = pd.read_excel(f"{GCS_FILE_PATH}/samtrans_ridership.xlsx")
santa_cruz_metro_ridership = pd.read_excel(f"{GCS_FILE_PATH}/santa_cruz_metro_ridership.xlsx")
sbmtd_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sbmtd_ridership.xlsx")
sdmts_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sdmts_ridership.xlsx")
sunline_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sunline_transit_ridership.xlsx")


In [5]:
def add_day_type(df, date_col, output_col='day_type'):
    """
    Classify each row as 'Weekday', 'Saturday', or 'Sunday' based on a specified date column.
    This is useful when daily datasets include labels such as 'Holiday' or when no day_type
    column exists. The function converts the date column, identifies the day of week, and
    standardizes the day-type classification accordingly. Either a start_date or end_date
    column can be used for datasets with daily reporting.
    """
    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    dow = df[date_col].dt.dayofweek  

    df[output_col] = np.select(
        condlist=[dow.eq(5), dow.eq(6)],
        choicelist=['Saturday', 'Sunday'],
        default='Weekday'
    )

    return df

In [6]:
# Counting number of days without holidays
def count_service_days(row):
    
    if str(row.day_type).lower() == "all":
        return (row.end_date - row.start_date).days + 1

    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum(d.weekday() < 5 for d in dates)

    elif row.day_type == "Saturday":
        return sum(d.weekday() == 5 for d in dates)

    elif row.day_type == "Sunday":
        return sum(d.weekday() == 6 for d in dates)

In [7]:
# Counting number of days with holidays
def count_service_days_wo_holidays(row):
    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum((d.weekday() < 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Saturday":
        return sum((d.weekday() == 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Sunday":
        return sum((d.weekday() == 6) and (d not in holiday_set) for d in dates)

In [8]:
def compute_averages(df, ridership_cols, group_cols):
    """
    Computes average of one or more ridership columns per group in long format.

    Parameters:
    - df: pandas DataFrame
    - ridership_cols: list of columns to average (e.g., ['daily_boardings', 'daily_alightings'])
    - group_cols: list of columns to group by (e.g., ['stop_name', 'day_type'])

    Returns:
    - DataFrame with columns: group_cols + average of each ridership column
    """
    
    if isinstance(ridership_cols, str):
        ridership_cols = [ridership_cols]  # allow single string input
    
    averaged = df.groupby(group_cols)[ridership_cols].mean().reset_index()
    
    # rename each column to indicate it's averaged
    averaged = averaged.rename(columns={col: f"average_{col}" for col in ridership_cols})
    
    return averaged

In [9]:
master_cols = [
    "organization_name",
    "stop_id",
    "stop_name",
    "stop_lat",
    "stop_lon",
    "day_type",
    "average_daily_boardings",
    "average_daily_alightings",
    "start_date",
    "end_date"
]

def standardize_columns(df, master_cols):
    
    # add missing columns
    for col in master_cols:
        if col not in df.columns:
            df[col] = None

    # keep only master columns and order them
    return df[master_cols]

In [10]:
bart_ridership = add_day_type(bart_ridership, date_col='start_date')

bart_ridership = bart_ridership.rename(columns={
    'Station Name': 'stop_name',
    'Stop ID': 'stop_id',
    'Date': 'date'
})

# Grouping columns
group_cols = [
    "date", "stop_id", "stop_name",
    "start_date", "end_date", "day_type"
]

# Sum boardings and alightings, with explicit output names
total_bart_ridership = (
    bart_ridership
    .groupby(group_cols, as_index=False)
    .agg(
        daily_boardings=("Entries", "sum"),
        daily_alightings=("Exits", "sum")
    )
)

# Set the ridership basis flag
total_bart_ridership["daily_ridership_basis"] = "reported_daily"


In [11]:
# Calculate average ridership per day type and stop
total_bart_ridership['daily_boardings'] = pd.to_numeric(total_bart_ridership['daily_boardings'], errors='coerce')
total_bart_ridership['daily_alightings'] = pd.to_numeric(total_bart_ridership['daily_alightings'], errors='coerce')

average_bart_ridership = compute_averages(
    df=total_bart_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id", "day_type", "stop_name"]
)

average_bart_ridership["organization_name"] = "San Francisco Bay Area Rapid Transit District"

average_bart_ridership[['start_date','end_date']] = [bart_ridership['start_date'].min(), bart_ridership['end_date'].max()]
# Remove last digit from all stop_id
average_bart_ridership['stop_id'] = average_bart_ridership['stop_id'].astype(str).str[:-1]

bart_final = standardize_columns(average_bart_ridership, master_cols)

bart_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
97,San Francisco Bay Area Rapid Transit District,90430,North Berkeley,None,None,Sunday,950.403846,949.269231,2024-10-01,2025-09-30
102,San Francisco Bay Area Rapid Transit District,90450,El Cerrito Del Norte,None,None,Saturday,2040.692308,2103.057692,2024-10-01,2025-09-30
40,San Francisco Bay Area Rapid Transit District,90210,Lake Merritt,None,None,Sunday,1359.153846,1477.057692,2024-10-01,2025-09-30
132,San Francisco Bay Area Rapid Transit District,90740,Oakland International Airport (OAK),None,None,Saturday,423.750000,323.673077,2024-10-01,2025-09-30
78,San Francisco Bay Area Rapid Transit District,90350,Pleasant Hill / Contra Costa Centre,None,None,Saturday,1108.596154,1070.865385,2024-10-01,2025-09-30


In [12]:
big_blue_bus_ridership.columns = big_blue_bus_ridership.columns.str.lower()
big_blue_bus_ridership['service_day'] = big_blue_bus_ridership['service_day'].str.strip().str.capitalize()


big_blue_bus_ridership = big_blue_bus_ridership.rename(columns={
    'service_day': 'day_type',
    'average_daily_boardings':'average_daily_boardings_period',
    'average_daily_alightings':'average_daily_alightings_period'    
})


# Sum boardings/alightings across directions within same stop & period
total_big_blue_bus_ridership = (
    big_blue_bus_ridership
    .groupby(['day_type',
              'stop_id','stop_name','stop_lat','stop_lon',
              'start_date','end_date'])
    .agg({
        'average_daily_boardings_period':'sum',
        'average_daily_alightings_period':'sum'
    })
    .reset_index()
)


In [13]:
big_blue_bus_ridership['start_date'] = pd.to_datetime(big_blue_bus_ridership['start_date'])
big_blue_bus_ridership['end_date'] = pd.to_datetime(big_blue_bus_ridership['end_date'])

total_big_blue_bus_ridership["n_days"] = total_big_blue_bus_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 4 different time periods
averaged_total_big_blue_bus_ridership = (
    total_big_blue_bus_ridership
    .groupby(['day_type', 'stop_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['average_daily_alightings_period'], weights=x['n_days'])
    }))
    .reset_index()
)


# Set the ridership basis flag
averaged_total_big_blue_bus_ridership["daily_ridership_basis"] = "reported_avg_daily"
averaged_total_big_blue_bus_ridership["organization_name"] = "Big Blue Bus"

averaged_total_big_blue_bus_ridership[['start_date','end_date']] = [big_blue_bus_ridership['start_date'].min(), big_blue_bus_ridership['end_date'].max()]

big_blue_bus_ridership_final = standardize_columns(averaged_total_big_blue_bus_ridership, master_cols)
big_blue_bus_ridership_final.sample(5)

/tmp/ipykernel_2651/1528211661.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
1880,Big Blue Bus,1253,MONTANA WB/20TH NS,34.036719,-118.489789,Weekday,7.897664,5.496865,2024-08-01,2025-11-30
1680,Big Blue Bus,2902,COLORADO EB/FRANKLIN FS,34.035331,-118.465279,Weekday,0.509368,7.910947,2024-08-01,2025-11-30
626,Big Blue Bus,2874,VIA MARINA SB/PANAY FS,33.978852,-118.459239,Saturday,13.360429,0.022078,2024-08-01,2025-11-30
1229,Big Blue Bus,3169,SAN VICENTE WB/FOXTAIL NS,34.036422,-118.506192,Sunday,0.400000,0.200000,2024-08-01,2025-11-30
42,Big Blue Bus,1102,4TH SB/BICKNELL FS,34.007841,-118.485558,Saturday,1.267229,3.631652,2024-08-01,2025-11-30


In [14]:
caltrain_ridership = caltrain_ridership.rename(columns={
    'Origin Station': 'stop_name',
    'Date Type': 'day_type',
    'Average Ridership':'average_daily_boardings_period',  
})

caltrain_ridership = caltrain_ridership[
    caltrain_ridership["day_type"] != "Holiday"
]

# Getting number of days and using US Federal Holidays
start_date_caltrain = '2023-11-01'
end_date_caltrain = '2025-07-31'

#Getting all the federal holidays between the start and end date
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=start_date_caltrain, end=end_date_caltrain)

# Setting holidays to calculate the number of weekdays, saturday and sunday without holidays 
holiday_set = set(holidays)

caltrain_ridership["start_date"] = pd.to_datetime(caltrain_ridership["start_date"])
caltrain_ridership["end_date"] = pd.to_datetime(caltrain_ridership["end_date"])

caltrain_ridership["n_days"] = caltrain_ridership.apply(count_service_days_wo_holidays, axis=1)

In [15]:
# Taking weighted average of the ridership across 20 months time periods
averaged_caltrain_ridership = (
    caltrain_ridership
    .groupby(['day_type','stop_name'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days'])
    }))
    .reset_index()
)

# Set the ridership basis flag
averaged_caltrain_ridership["daily_ridership_basis"] = "reported_avg_daily"
averaged_caltrain_ridership["organization_name"] = "Caltrain"

averaged_caltrain_ridership[['start_date','end_date']] = [caltrain_ridership['start_date'].min(), caltrain_ridership['end_date'].max()]

caltrain_ridership_final = standardize_columns(averaged_caltrain_ridership, master_cols)

caltrain_ridership_final.sample(5)

/tmp/ipykernel_2651/3722569284.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
59,Caltrain,None,Tamien,None,None,Sunday,43.960035,None,2023-11-01,2025-07-31
34,Caltrain,None,Broadway,None,None,Sunday,66.493661,None,2023-11-01,2025-07-31
26,Caltrain,None,Santa Clara,None,None,Saturday,513.810691,None,2023-11-01,2025-07-31
4,Caltrain,None,Broadway,None,None,Saturday,80.031640,None,2023-11-01,2025-07-31
82,Caltrain,None,San Francisco,None,None,Weekday,5656.512364,None,2023-11-01,2025-07-31


In [16]:
culver_citybus_ridership["daily_ridership_basis"] = "reported_avg_daily"



culver_citybus_ridership = culver_citybus_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
})

group_cols = [
     "stop_id", "stop_name", "day_type", "daily_ridership_basis"]

# Sum AVG On and AVG Off
sum_culver_citybus_ridership = (culver_citybus_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("AVG On", "sum"),
        average_daily_alightings=("AVG Off", "sum")
    )
)

sum_culver_citybus_ridership["organization_name"] = "Culver City Bus"

sum_culver_citybus_ridership[['start_date','end_date']] = [culver_citybus_ridership['start_date'].min(), culver_citybus_ridership['end_date'].max()]

culver_citybus_final = standardize_columns(sum_culver_citybus_ridership, master_cols)

culver_citybus_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
373,Culver City Bus,349,Beverly Glen Blvd/Pico Blvd,None,None,Sunday,0.8,1.2,2025-07-14,2025-08-25
540,Culver City Bus,441,Jefferson Blvd/Machado Rd,None,None,Sunday,2.1,1.9,2025-07-14,2025-08-25
155,Culver City Bus,152,Washington Blvd/Jasmine Ave,None,None,Weekday,18.3,4.2,2025-07-14,2025-08-25
487,Culver City Bus,405,Jefferson Blvd/Westlawn Ave,None,None,Weekday,4.0,0.4,2025-07-14,2025-08-25
631,Culver City Bus,634,Sepulveda Blvd/Richland Ave,None,None,Saturday,5.7,8.6,2025-07-14,2025-08-25


In [17]:
# Add day_type from the date column
foothill_transit_ridership = add_day_type(foothill_transit_ridership, date_col="date")

foothill_transit_ridership = foothill_transit_ridership.rename(columns={
    'stop_code': 'stop_id',
})

# Grouping columns
group_cols = [
    "date", "stop_id", "stop_lat", "stop_lon",
    "start_date", "end_date", "day_type"
]

# Sum boardings and alightings, with explicit output names
total_ridership_foothill = (
    foothill_transit_ridership
    .groupby(group_cols, as_index=False)
    .agg(
        daily_boardings=("boardings", "sum"),
        daily_alightings=("alightings", "sum")
    )
)

# Set the ridership basis flag
total_ridership_foothill["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_ridership_foothill['daily_boardings'] = pd.to_numeric(total_ridership_foothill['daily_boardings'], errors='coerce')
total_ridership_foothill['daily_alightings'] = pd.to_numeric(total_ridership_foothill['daily_alightings'], errors='coerce')

average_foothill_ridership = compute_averages(
    df=total_ridership_foothill,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=[ "stop_id", 
                "stop_lat", "stop_lon", "day_type"]
)

average_foothill_ridership["organization_name"] = "Foothill Transit"

average_foothill_ridership[['start_date','end_date']] = [foothill_transit_ridership['start_date'].min(), foothill_transit_ridership['end_date'].max()]
foothill_final = standardize_columns(average_foothill_ridership, master_cols)

foothill_final.sample(5)


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3020,Foothill Transit,2265,None,34.128586,-117.885967,Saturday,13.711538,5.730769,2024-07-01,2025-06-30
3335,Foothill Transit,2511,None,34.053702,-117.954160,Weekday,27.819923,14.819923,2024-07-01,2025-06-30
4358,Foothill Transit,3233,None,34.075318,-117.889839,Saturday,4.076923,3.211538,2024-07-01,2025-06-30
618,Foothill Transit,765,None,34.047677,-117.908293,Weekday,16.808429,5.574713,2024-07-01,2025-06-30
2208,Foothill Transit,1714,None,34.061569,-117.787339,Sunday,7.784314,1.568627,2024-07-01,2025-06-30


In [18]:
# Add day_type from the date column
fresno_area_express_ridership = add_day_type(fresno_area_express_ridership, date_col="Date")

fresno_area_express_ridership["daily_ridership_basis"] = "reported_daily"


fresno_area_express_ridership = fresno_area_express_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopLabel': 'stop_name',
    'ProjectedBoarding': 'daily_boardings',
    'ProjectedAlighting': 'daily_alightings'
})

# Calculate average ridership per day type and stop
fresno_area_express_ridership['daily_boardings'] = pd.to_numeric(fresno_area_express_ridership['daily_boardings'], errors='coerce')
fresno_area_express_ridership['daily_alightings'] = pd.to_numeric(fresno_area_express_ridership['daily_alightings'], errors='coerce')

average_fresno_area_express_ridership = compute_averages(
    df=fresno_area_express_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id",  "stop_name", "day_type"]
)

average_fresno_area_express_ridership["organization_name"] = "Fresno County"

average_fresno_area_express_ridership[['start_date','end_date']] = [fresno_area_express_ridership['start_date'].min(), fresno_area_express_ridership['end_date'].max()]

fresno_final = standardize_columns(average_fresno_area_express_ridership, master_cols)

fresno_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3421,Fresno County,1597,SE SHIELDS - CALLISCH,None,None,Sunday,0.179141,0.065096,2024-09-01,2025-08-31
211,Fresno County,126,MAPLE - KINGS CANYON - EL MONTE,None,None,Saturday,1.078233,8.210642,2024-09-01,2025-08-31
578,Fresno County,293,NW SHAW - PALM,None,None,Sunday,16.362652,8.915165,2024-09-01,2025-08-31
4394,Fresno County,2304,NW DAKOTA - NINTH,None,None,Saturday,7.973982,1.544657,2024-09-01,2025-08-31
3894,Fresno County,1808,NE ASHLAN - BARTON,None,None,Saturday,2.318602,0.719954,2024-09-01,2025-08-31


In [19]:
gold_coast_transit_ridership = gold_coast_transit_ridership.rename(columns={
    'day_of_week': 'day_type',
    'lat': 'stop_lat',
    'lon': 'stop_lon'
})

group_cols = ["day_type", "stop_id", "stop_name",  "stop_lat", "stop_lon", "start_date", "end_date"]

# Sum AVG On 
total_gold_coast_transit_ridership = (gold_coast_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("total_on", "sum"),
        daily_alightings=("total_off", "sum"),
    )
)

total_gold_coast_transit_ridership['start_date'] = pd.to_datetime(total_gold_coast_transit_ridership['start_date'])
total_gold_coast_transit_ridership['end_date'] = pd.to_datetime(total_gold_coast_transit_ridership['end_date'])

total_gold_coast_transit_ridership["n_days"] = total_gold_coast_transit_ridership.apply(count_service_days, axis=1)

In [20]:
# Taking weighted average of the ridership across 4 different time periods
averaged_gold_coast_transit_ridership = (
    total_gold_coast_transit_ridership
    .groupby(['day_type', 'stop_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['daily_boardings'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['daily_alightings'], weights=x['n_days'])
    }))
    .reset_index()
)

averaged_gold_coast_transit_ridership["organization_name"] = "Gold Coast Transit"

averaged_gold_coast_transit_ridership[['start_date','end_date']] = [gold_coast_transit_ridership['start_date'].min(), gold_coast_transit_ridership['end_date'].max()]
golden_coast_transit_final = standardize_columns(averaged_gold_coast_transit_ridership, master_cols)

golden_coast_transit_final.sample(5)

/tmp/ipykernel_2651/1215055102.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
499,Gold Coast Transit,THOKAL2,Thompson & Kalorama,34.278211,-119.286686,Weekday,16.0,7.0,2025-05-01,2025-05-31
362,Gold Coast Transit,ROSWOO1,Rose & Wooley,34.189541,-119.160032,Weekday,1.0,3.0,2025-05-01,2025-05-31
336,Gold Coast Transit,ROSVEN2,Rose & Auto Center,34.228359,-119.157635,Weekday,2.0,0.0,2025-05-01,2025-05-31
504,Gold Coast Transit,THOSEA1,Thompson & Seaward,34.273788,-119.266868,Weekday,42.0,16.0,2025-05-01,2025-05-31
83,Gold Coast Transit,CSTHEM2,C & Hemlock,34.181350,-119.181220,Weekday,21.0,11.0,2025-05-01,2025-05-31


In [21]:
# Add day_type from the date column
golden_gate_park_shuttle_ridership = add_day_type(golden_gate_park_shuttle_ridership, date_col="Date")

group_cols = [
    "Date", "day_type", "Stop", "Stop ID", "start_date", "end_date"]

# Sum AVG On 
total_golden_gate_park_shuttle_ridership = (golden_gate_park_shuttle_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Ridership", "sum"),
    )
)

total_golden_gate_park_shuttle_ridership["daily_ridership_basis"] = "reported_daily"

total_golden_gate_park_shuttle_ridership = total_golden_gate_park_shuttle_ridership.rename(columns={
    'Stop': 'stop_name',
    'Date': 'date',
    'Stop ID': 'stop_id'
})

# Calculate average ridership per day type and stop
total_golden_gate_park_shuttle_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_park_shuttle_ridership['daily_boardings'], errors='coerce')
average_golden_gate_park_shuttle_ridership = compute_averages(
    df=total_golden_gate_park_shuttle_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["stop_name", "day_type", "stop_id"]
)


average_golden_gate_park_shuttle_ridership["organization_name"] = "Golden Gate Park Shuttle"

average_golden_gate_park_shuttle_ridership[['start_date','end_date']] = [golden_gate_park_shuttle_ridership['start_date'].min(), 
                                                                         golden_gate_park_shuttle_ridership['end_date'].max()]
golden_gate_parkshuttle_final = standardize_columns(average_golden_gate_park_shuttle_ridership, master_cols)

golden_gate_parkshuttle_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
12,Golden Gate Park Shuttle,7609,Academy of Sciences,None,None,Saturday,13.288462,None,2024-07-01,2025-06-30
6,Golden Gate Park Shuttle,7611,8th Ave EB,None,None,Saturday,10.673077,None,2024-07-01,2025-06-30
9,Golden Gate Park Shuttle,7610,8th Ave WB,None,None,Saturday,11.961538,None,2024-07-01,2025-06-30
13,Golden Gate Park Shuttle,7609,Academy of Sciences,None,None,Sunday,14.903846,None,2024-07-01,2025-06-30
14,Golden Gate Park Shuttle,7609,Academy of Sciences,None,None,Weekday,12.881226,None,2024-07-01,2025-06-30


In [22]:
# Add day_type from the date column
golden_gate_transit_ridership = add_day_type(golden_gate_transit_ridership, date_col="date")

golden_gate_transit_ridership = golden_gate_transit_ridership.rename(columns={
    'STOP_NUMBER': 'stop_id',
    'STOP_NAME': 'stop_name',
})

# Remove everything in parentheses (including the parentheses) at the end of the string
golden_gate_transit_ridership['stop_name'] = golden_gate_transit_ridership['stop_name'].str.replace(r'\s*\(.*\)$', '', regex=True)

group_cols = [
    "day_type", "stop_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
total_golden_gate_transit_ridership = (golden_gate_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("BOARDINGS", "sum"),
        daily_alightings=("ALIGHTINGS", "sum")
    )
)

total_golden_gate_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_golden_gate_transit_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_boardings'], errors='coerce')
total_golden_gate_transit_ridership['daily_alightings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_alightings'], errors='coerce')

average_golden_gate_transit_ridership = compute_averages(
    df=total_golden_gate_transit_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id", "stop_name", "day_type"]
)

average_golden_gate_transit_ridership["organization_name"] = "Golden Gate Transit"
average_golden_gate_transit_ridership[['start_date','end_date']] = [golden_gate_transit_ridership['start_date'].min(), 
                                                                         golden_gate_transit_ridership['end_date'].max()]

golden_gate_transit_final = standardize_columns(average_golden_gate_transit_ridership, master_cols)

golden_gate_transit_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
34,Golden Gate Transit,40032,Van Ness Ave & Union St,None,None,Sunday,51.750000,14.500000,2025-09-01,2025-09-30
429,Golden Gate Transit,41091,Francisco Blvd E & Grange Way,None,None,Sunday,1.250000,2.500000,2025-09-01,2025-09-30
95,Golden Gate Transit,40093,Bridgeway & Nevada St,None,None,Sunday,0.500000,0.750000,2025-09-01,2025-09-30
433,Golden Gate Transit,41092,Francisco Blvd E & Morphew St,None,None,Weekday,2.090909,0.954545,2025-09-01,2025-09-30
402,Golden Gate Transit,41000,Hwy 101 & Alexander Ave,None,None,Sunday,16.750000,43.500000,2025-09-01,2025-09-30


In [23]:
long_beach_transit_ridership["daily_ridership_basis"] = "reported_avg_daily"

group_cols = [
    "DayType", "StopID", "StopName", "end_date", "start_date", "daily_ridership_basis"]

# Sum AVG On and AVG Off
total_long_beach_transit_ridership = (long_beach_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("Boardings", "sum"),
        average_daily_alightings=("Alightings", "sum")
    )
)

total_long_beach_transit_ridership = total_long_beach_transit_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopName': 'stop_name',
    'DayType': 'day_type',
})

total_long_beach_transit_ridership["organization_name"] = "Long Beach Transit"

total_long_beach_transit_ridership[['start_date','end_date']] = [long_beach_transit_ridership['start_date'].min(), 
                                                                         long_beach_transit_ridership['end_date'].max()]

long_beach_transit_final = standardize_columns(total_long_beach_transit_ridership, master_cols)

long_beach_transit_final.sample(5)


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3170,Long Beach Transit,1902,Norwalk & Flora Vista NE,None,None,Sunday,4.986962,6.632944,2024-07-01,2025-06-30
356,Long Beach Transit,434,Pacific & 20th NE,None,None,Saturday,6.385321,11.161290,2024-07-01,2025-06-30
2790,Long Beach Transit,1390,Lakewood & Willow SW,None,None,Sunday,18.949916,6.961785,2024-07-01,2025-06-30
52,Long Beach Transit,63,Long Beach Blvd & 8th SW,None,None,Saturday,0.000000,60.038409,2024-07-01,2025-06-30
781,Long Beach Transit,1066,Bellflower & Oak SW,None,None,Saturday,20.759690,4.787729,2024-07-01,2025-06-30


In [24]:
octa_ridership = octa_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',

})

octa_ridership["stop_name"] = octa_ridership["stop_name"].str.split("-", n=1).str[1]

group_cols = [
    "day_type", "stop_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
octa_ridership_sum = (octa_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("APC Boarding", "sum"),
        average_daily_alightings=("APC Alighting", "sum")
    )
)

octa_ridership_sum.loc[
    octa_ridership_sum['day_type'].str.strip().str.lower() == "weekday",
    'day_type'
] = "Weekday"

octa_ridership_sum["organization_name"] = "Orange County Transportation Authority"

octa_ridership_sum[['start_date','end_date']] = [octa_ridership['start_date'].min(), 
                                                                         octa_ridership['end_date'].max()]

octa_final = standardize_columns(octa_ridership_sum, master_cols)

octa_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2921,Orange County Transportation Authority,4862,BALBOA-41ST,None,None,Weekday,0,4,2025-02-04,2025-02-04
742,Orange County Transportation Authority,1074,17TH-IRVINE,None,None,Weekday,3,41,2025-02-04,2025-02-04
3266,Orange County Transportation Authority,5405,GLASSELL-LINCOLN,None,None,Weekday,8,43,2025-02-04,2025-02-04
1582,Orange County Transportation Authority,2383,HARBOR-TRASK,None,None,Weekday,47,44,2025-02-04,2025-02-04
3714,Orange County Transportation Authority,6215,EDINGER-LYON,None,None,Weekday,39,4,2025-02-04,2025-02-04


In [25]:
omni_trans_ridership= omni_trans_ridership.rename(columns={
    'Stop Name': 'stop_name',
    'avg_boardings': 'average_daily_boardings',
    'avg_alightings': 'average_daily_alightings',
    'Stop Id': 'stop_id'
})

omni_trans_ridership["organization_name"] = "Omnitrans"

omni_trans_ridership_filtered = omni_trans_ridership[['stop_name', 'stop_id',  'average_daily_boardings', 'average_daily_alightings', 'organization_name', 'day_type']]

omni_trans_ridership_filtered['stop_id'] = omni_trans_ridership_filtered['stop_id'].apply(
    lambda x: None if pd.isna(x) else int(x)
)

omni_trans_ridership_filtered[['start_date','end_date']] = [omni_trans_ridership['start_date'].min(), 
                                                                         omni_trans_ridership['end_date'].max()]
omni_final = standardize_columns(omni_trans_ridership_filtered, master_cols)

omni_final.sample(5)

/tmp/ipykernel_2651/3069775595.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omni_trans_ridership_filtered['stop_id'] = omni_trans_ridership_filtered['stop_id'].apply(
/tmp/ipykernel_2651/3069775595.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omni_trans_ridership_filtered[['start_date','end_date']] = [omni_trans_ridership['start_date'].min(),
/tmp/ipykernel_2651/3069775595.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_in

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3874,Omnitrans,6508.0,Eucalyptus @ Merrill,None,None,all,0.564384,1.438356,2023-07-01,2026-06-30
478,Omnitrans,6296.0,Baseline @ Lilac,None,None,all,3.005464,3.218579,2023-07-01,2026-06-30
4416,Omnitrans,551.0,MILLIKEN @ 6TH ST,None,None,all,0.263014,0.323288,2023-07-01,2026-06-30
652,Omnitrans,6519.0,Mill @ K,None,None,all,1.226776,1.390710,2023-07-01,2026-06-30
636,Omnitrans,6502.0,Merrill @ Larch,None,None,all,1.816940,1.584699,2023-07-01,2026-06-30


In [26]:
riverside_transit_ridership = add_day_type(riverside_transit_ridership, date_col='date')

group_cols = [
    "date", "Stop ID", "start_date", 
    "end_date", "day_type"]

# Sum AVG On and AVG Off
total_riverside_transit_ridership = (riverside_transit_ridership.groupby(
    group_cols, as_index=False).agg(
    daily_boardings = ('ridership', 'sum'))
)

total_riverside_transit_ridership = total_riverside_transit_ridership.rename(columns={
    'Stop ID': 'stop_id'

})

In [27]:
total_riverside_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_riverside_transit_ridership['daily_boardings'] = pd.to_numeric(total_riverside_transit_ridership['daily_boardings'], errors='coerce')
average_riverside_transit_ridership = compute_averages(
    df=total_riverside_transit_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["stop_id", "day_type"]
)

average_riverside_transit_ridership["organization_name"] = "Riverside Transit"

average_riverside_transit_ridership[['start_date','end_date']] = [riverside_transit_ridership['start_date'].min(), 
                                                                         riverside_transit_ridership['end_date'].max()]

riverside_final = standardize_columns(average_riverside_transit_ridership, master_cols)

riverside_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
5813,Riverside Transit,3730,None,None,None,Sunday,5.777778,None,2025-01-01,2025-10-31
2227,Riverside Transit,1861,None,None,None,Saturday,2.724138,None,2025-01-01,2025-10-31
3669,Riverside Transit,2456,None,None,None,Weekday,9.875676,None,2025-01-01,2025-10-31
971,Riverside Transit,1382,None,None,None,Sunday,1.724138,None,2025-01-01,2025-10-31
6096,Riverside Transit,4747,None,None,None,Saturday,22.380952,None,2025-01-01,2025-10-31


In [28]:
group_cols = [
    "stop_id", "stop_lat", "stop_lon", "stop_name", "day_type"]

# Combining across directions
sacrt_bus_ridership_sum = (sacrt_bus_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("average_boardings", "sum"),
        average_daily_alightings=("average_alightings", "sum")
    )
)

sacrt_bus_ridership_sum.loc[
    sacrt_bus_ridership_sum['day_type'].str.strip().str.lower() == "weekday",
    'day_type'
] = "Weekday"

# Replace day_type
sacrt_bus_ridership_sum["day_type"] = sacrt_bus_ridership_sum["day_type"].replace("daily", "all")

sacrt_bus_ridership_sum["organization_name"] = "SacRT Bus"

sacrt_bus_ridership_sum[['start_date','end_date']] = [sacrt_bus_ridership['start_date'].min(), 
                                                                         sacrt_bus_ridership['end_date'].max()]

sacrt_bus_final = standardize_columns(sacrt_bus_ridership_sum, master_cols)

sacrt_bus_final.sample(5)


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2743,SacRT Bus,4710,TRUXEL RD & GATEWAY PARK BLVD (NB),38.638125,-121.504456,all,20,86,2023-09-01,2023-12-31
857,SacRT Bus,1156,ARDEN WAY & PROFESSIONAL DR (WB),38.595890,-121.387581,Weekday,1,0,2023-09-01,2023-12-31
1640,SacRT Bus,2235,FREEPORT BLVD & SUTTERVILLE RD (NB),38.538648,-121.492595,all,3,6,2023-09-01,2023-12-31
1833,SacRT Bus,2472,21ST ST & 68TH AVE (NB),38.489349,-121.491888,Weekday,2,0,2023-09-01,2023-12-31
2673,SacRT Bus,4262,RIVERSIDE BLVD & PARK RIVIERA WAY (EB),38.503182,-121.547050,Weekday,0,0,2023-09-01,2023-12-31


In [29]:
samtrans_ridership = add_day_type(samtrans_ridership, date_col='APC Date')

samtrans_ridership = samtrans_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Lat': 'stop_lat',
    'Lon': 'stop_lon',
    'APC Date': 'date',
    'Ons': 'daily_boardings',
    'Offs': 'daily_alightings',
})

samtrans_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
samtrans_ridership['daily_boardings'] = pd.to_numeric(samtrans_ridership['daily_boardings'], errors='coerce')
samtrans_ridership['daily_alightings'] = pd.to_numeric(samtrans_ridership['daily_alightings'], errors='coerce')

average_samtrans_ridership = compute_averages(
    df=samtrans_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id", "stop_name", "stop_lat", "stop_lon", "day_type"]
)

average_samtrans_ridership["organization_name"] = "Samtrans"

average_samtrans_ridership[['start_date','end_date']] = [samtrans_ridership['start_date'].min(), 
                                                                         samtrans_ridership['end_date'].max()]

samtrans_final = standardize_columns(average_samtrans_ridership, master_cols)

samtrans_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
331,Samtrans,313005,Etheldore St & Marine Blvd,37.524913,-122.508514,Sunday,0.5,1.000000,2025-08-01,2025-08-31
2040,Samtrans,335508,McDonnell Rd & San Bruno Ave,37.633064,-122.397644,Saturday,2.0,1.333333,2025-08-01,2025-08-31
2325,Samtrans,341026,E 3rd Ave & S Humboldt St,37.569847,-122.315994,Sunday,1.2,10.000000,2025-08-01,2025-08-31
1385,Samtrans,333004,El Camino Real & Colma Blvd,37.680568,-122.461601,Sunday,9.0,12.428571,2025-08-01,2025-08-31
94,Samtrans,311085,Hwy 1 & Rockaway Beach Ave,37.608375,-122.495674,Saturday,1.8,17.200000,2025-08-01,2025-08-31


In [30]:
group_cols = [
    "Stop ID",  "Stop Name", "day_type", "start_date", 
    "end_date"]

# Sum AVG On and AVG Off
total_sdmts_ridership = (sdmts_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("Average On", "sum"),
        average_daily_alightings=("Average Off", "sum")
    )
)


total_sdmts_ridership = total_sdmts_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
})

total_sdmts_ridership["daily_ridership_basis"] = "reported_avg_daily"

total_sdmts_ridership["organization_name"] = "SDMTS"

total_sdmts_ridership[['start_date','end_date']] = [sdmts_ridership['start_date'].min(), 
                                                                         sdmts_ridership['end_date'].max()]

sdmts_final = standardize_columns(total_sdmts_ridership, master_cols)

sdmts_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
8331,SDMTS,60384,Silver Strand Bl & Leyte Rd,None,None,Saturday,3.259126,0.968160,2024-09-01,2025-01-25
2795,SDMTS,11730,Highland Av & 14th St,None,None,Sunday,23.059820,12.298170,2024-09-01,2025-01-25
8965,SDMTS,75010,H Street Station,None,None,Sunday,1902.954563,2570.462045,2024-09-01,2025-01-25
9007,SDMTS,75025,Arnele Avenue Station,None,None,Sunday,489.233333,164.219841,2024-09-01,2025-01-25
2190,SDMTS,11337,National Av & 38th St,None,None,Saturday,69.498465,21.358145,2024-09-01,2025-01-25


In [31]:
santa_cruz_metro_ridership = santa_cruz_metro_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Boardings': 'average_daily_boardings',
    'Alightings': 'average_daily_alightings'    
})

santa_cruz_metro_ridership["organization_name"] = "Santa Cruz Metro"

santa_cruz_metro_ridership[['start_date','end_date']] = [santa_cruz_metro_ridership['start_date'].min(), 
                                                                         santa_cruz_metro_ridership['end_date'].max()]

santa_cruz_final = standardize_columns(santa_cruz_metro_ridership, master_cols)

santa_cruz_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
612,Santa Cruz Metro,1731,Scotts Valley Dr + Bean Creek Rd (Middle School),None,None,all,1.986301,14.232877,2024-07-01,2025-06-30
519,Santa Cruz Metro,2480,Neilson + Airport Blvd (Wats. Hospital),None,None,all,19.778082,17.901370,2024-07-01,2025-06-30
451,Santa Cruz Metro,1130,Kralj Dr + Lawrence Ave,None,None,all,0.000000,0.005479,2024-07-01,2025-06-30
648,Santa Cruz Metro,1795,Soquel Ave + Frederick,None,None,all,47.923288,26.663014,2024-07-01,2025-06-30
28,Santa Cruz Metro,1922,7th Ave + Capitola Rd,None,None,all,1.183562,1.506849,2024-07-01,2025-06-30


In [32]:
sbmtd_ridership = sbmtd_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'avg_boardings': 'daily_boardings',
    'avg_ridership': 'daily_alightings'    
})


sbmtd_ridership["n_days"] = sbmtd_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 12 months
averaged_sbmtd_ridership = (
    sbmtd_ridership
    .groupby(['stop_id', 'stop_name', 'day_type'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['Ridership by Stop_Boardings'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['Ridership by Stop_Alightings'], weights=x['n_days'])
    }))
    .reset_index()
)



averaged_sbmtd_ridership["organization_name"] = "SBMTD"

averaged_sbmtd_ridership[['start_date','end_date']] = [sbmtd_ridership['start_date'].min(), 
                                                                         sbmtd_ridership['end_date'].max()]

sbmtd_final = standardize_columns(averaged_sbmtd_ridership, master_cols)

sbmtd_final.sample(5)

/tmp/ipykernel_2651/1734702637.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
76,SBMTD,92,Chapala & Haley,None,None,all,223.761644,363.986301,2024-11-01,2025-10-31
417,SBMTD,498,County Health & Veteran Clinic,None,None,all,30.098630,108.400000,2024-11-01,2025-10-31
461,SBMTD,569,State & Mason,None,None,all,33.139726,4.619178,2024-11-01,2025-10-31
306,SBMTD,359,San Ysidro & San Leandro,None,None,all,16.301370,9.238356,2024-11-01,2025-10-31
36,SBMTD,49,Hollister & Pacific Oaks,None,None,all,211.671233,259.449315,2024-11-01,2025-10-31


In [33]:
all_ridership = pd.concat([
    bart_final,
    big_blue_bus_ridership_final,
    caltrain_ridership_final,
    culver_citybus_final,
    foothill_final,
    fresno_final,
    golden_coast_transit_final,
    golden_gate_parkshuttle_final,
    golden_gate_transit_final,
    long_beach_transit_final,
    octa_final,
    omni_final,
    riverside_final,
    sacrt_bus_final,
    samtrans_final,
    sdmts_final,
    santa_cruz_final,
    sbmtd_final,
], ignore_index=True)

/tmp/ipykernel_2651/1180593662.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_ridership = pd.concat([


In [34]:
all_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55521 entries, 0 to 55520
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   organization_name         55521 non-null  object        
 1   stop_id                   55111 non-null  object        
 2   stop_name                 44301 non-null  object        
 3   stop_lat                  15342 non-null  float64       
 4   stop_lon                  15342 non-null  float64       
 5   day_type                  55521 non-null  object        
 6   average_daily_boardings   55521 non-null  float64       
 7   average_daily_alightings  49139 non-null  float64       
 8   start_date                55521 non-null  datetime64[ns]
 9   end_date                  55521 non-null  datetime64[ns]
dtypes: datetime64[ns](2), float64(4), object(4)
memory usage: 4.2+ MB


In [35]:
all_ridership.to_csv(f"{GCS_FILE_PATH}/AHSC_2026/ridership_all.csv", index=False)